<a href="https://colab.research.google.com/github/aartinangare1745/GuardPrompts/blob/main/Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#load datasets

**load dataset**
**NeCent(11m rows)**
drop extra column
rename column
starified sampling by cateogory of attack- 30k sample
remove duplicate- exact match by drop duplicate
final_Necent.csv


**MindGard**
drop extra column
rename column
starified sampling by cateogory of attack- 5k sample
drop duplicate
spilt into two dataset with label 0 for mormal and 1 for modified prompt.merge back into one column.
final_Mindgard.csv

**DeepSet**
add attack_type = unknown
rename columns
drop duplicate
final_Deepset


Merge all thre into final prompt_data


token access for authentication : huggingface

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab secrets
hf_token = userdata.get('HF_Token')

# Login to Hugging Face
login(token=hf_token)

print("Successfully logged in to Hugging Face.")

load all dataset from huggingface


In [ ]:
from datasets import load_dataset

Necent=load_dataset('Necent/llm-jailbreak-prompt-injection-dataset')
Mindgard=load_dataset("Mindgard/evaded-prompt-injection-and-jailbreak-samples")
Deepset=load_dataset("deepset/prompt-injections")

In [ ]:
Mindgard

In [ ]:
Necent=Necent['train'].to_pandas()
Mindgard=Mindgard['train'].to_pandas()
Deepset=Deepset['train'].to_pandas()

sampling necent data 11m to 30k

In [ ]:
from sklearn.model_selection import train_test_split

# Fill NaN values if any remain
Necent['category_clean'] = Necent['category'].fillna('unknown')

# Keep categories with 2 or more rows
valid_categories = Necent['category_clean'].value_counts()
valid_categories = valid_categories[valid_categories >= 2].index
data_filtered = Necent[Necent['category_clean'].isin(valid_categories)]

#keeping only 30k rows
target_sample=30000
sample_fraction=target_sample/len(data_filtered)

# Stratified sampling on filtered data
_, Necent_sample = train_test_split(
    data_filtered,
    test_size=sample_fraction,
    stratify=data_filtered['category_clean'],
    random_state=42,
    shuffle=True,
)

# Optional: drop the temporary column
Necent_sample = Necent_sample.drop(columns=['category_clean'])

In [ ]:
len(Necent)

In [ ]:
len(Necent_sample)

In [ ]:
Necent.head(5)

In [ ]:
Necent_sample=Necent_sample.drop(columns=[
    "response","model_name","category","is_dangerous","prompt_harmful",
    "source","language","response_harmful","response_refusal",
    "attack_technique"
])
necent_sample.rename(columns={"prompt_adversarial":"label"},inplace=True)

In [ ]:
Necent_sample.rename(columns={"prompt_adversarial":"label"},inplace=True)

In [ ]:
Necent_sample

In [ ]:
Necent_sample=Necent_sample.drop_duplicates()

In [ ]:
len(Necent_sample)

 High-Speed MinHash LSH :
This technique hashes similar strings into the same buckets,dropping rows down to unique/near-unique prompts in 1 to 2 minutes.

In [ ]:
Necent_exact = Necent_sample.drop_duplicates(
    subset=["prompt"]
).reset_index(drop=True)

print("Original:", len(Necent_sample))
print("After exact dedup:", len(Necent_exact))

In [ ]:
!pip install datasketch

import re
from datasketch import MinHash, MinHashLSH

# Tokenizer used ONLY in memory to check similarity (doesn't touch your actual DataFrame)
def get_tokens(text):
    text = re.sub(r'[^\w\s]', '', str(text).lower())
    return set(text.split())

# 1. Initialize MinHash LSH (0.85 means ~85%+ similar prompts are treated as duplicates)
lsh = MinHashLSH(threshold=0.85, num_perm=128)

unique_indices = []

# 2. Iterate through your exact necent prompts
for idx, prompt in enumerate(Necent_sample['prompt']):
    tokens = get_tokens(prompt)
    if not tokens:
        continue

    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf-8'))

    # Check if a near-duplicate has already been seen
    result = lsh.query(m)
    if not result:
        lsh.insert(f"idx_{idx}", m)
        unique_indices.append(idx)

# 3. Pull the EXACT original rows (unmodified, full formatting intact)
necent_dedup = Necent_sample.iloc[unique_indices].reset_index(drop=True)

print(f"Original rows: {len(Necent_sample)} | Deduplicated rows: {len(necent_dedup)}")

In [ ]:
Mindgard=load_dataset("Mindgard/evaded-prompt-injection-and-jailbreak-samples")

In [ ]:
Mindgard

In [ ]:
Mindgard=Mindgard['train'].to_pandas()

sample mindgard

In [ ]:
from sklearn.model_selection import train_test_split
# Stratified sampling on attack type
_, Mindgard_sample= train_test_split(
    Mindgard,
    test_size=0.44,
    stratify=Mindgard["attack_name"],
    random_state=42,
    shuffle=True,
)

In [ ]:
len(Mindgard_sample)

In [ ]:
Mind_exact = Mindgard_sample.drop_duplicates(
    subset=["modified_sample"]
).reset_index(drop=True)

print("Original:", len(Mindgard_sample))
print("After exact dedup:", len(Mind_exact))

In [ ]:
mind1=Mind_exact.iloc[:,:2]
mind1["label"]=0
mind1.rename(columns={"original_sample":"prompt"},inplace=True)
len(mind1)

In [ ]:
mind1=mind1.drop_duplicates()
len(mind1)

In [ ]:
mind2=Mind_exact[["attack_name","modified_sample"]]
mind2["label"]=1

In [ ]:
mind2.rename(columns={"modified_sample":"prompt",},inplace=True)

In [ ]:
import pandas as pd
final_mind=pd.concat([mind1,mind2])
final_mind.rename(columns={"attack_name":"attack_type"},inplace=True)

In [ ]:
final_mind

In [ ]:
add attack_type column to deepset

In [ ]:
final_mind.rename(columns:{"prompt_type":"attack_type"})
Necent_sample.rename(columns:{"prompt_type":"attack_type"})
Deepset["attack_type"]="Unknown"